# Speculative Decoding in vLLM: A Complete Guide to Faster LLM Inference

---

**Vizuara - LLM Inference Engineering Course**  
**Lecture 09: Speculative Decoding -- Production Benchmarks with vLLM**

---

> *"The best optimization is the one that gives you free speed without changing a single output token."*

In this notebook, we go from theory to production. We benchmark **four speculative decoding strategies** using [vLLM](https://github.com/vllm-project/vllm) -- the industry-standard LLM serving engine -- on an **NVIDIA H100 80GB GPU**.

**What you'll learn:**
- Why standard autoregressive decoding leaves GPU compute on the table
- Five families of speculative decoding (Standard, N-gram, Suffix, Medusa, EAGLE)
- How to enable each method in vLLM with a single config flag
- Real benchmark numbers on H100 showing up to **2x speedup** with EAGLE-3
- When speculative decoding helps -- and when it actually hurts

## Table of Contents

| # | Section | What You'll Learn |
|---|---------|-------------------|
| 1 | [The Problem](#1-the-problem-autoregressive-decoding-is-memory-bound) | Why LLM generation underutilizes GPUs |
| 2 | [The Solution](#2-the-solution-speculate-then-verify) | The draft-and-verify paradigm |
| 3 | [Five Methods](#3-five-methods-of-speculative-decoding) | N-gram, Suffix, Draft Model, Medusa, EAGLE |
| 4 | [The Math](#4-the-math-why-output-quality-is-preserved) | Rejection sampling guarantees |
| 5 | [vLLM Setup](#5-vllm-setup-and-configuration) | How to enable speculation in vLLM |
| 6 | [Live Benchmarks](#6-live-benchmarks-h100-gpu) | Head-to-head comparisons on H100 |
| 7 | [Results Analysis](#7-results-analysis) | Benchmark table and analysis |
| 8 | [Under the Hood](#8-under-the-hood-code-walkthrough) | Implementing basic speculative sampling from scratch |
| 9 | [Production Guide](#9-production-guide-when-to-use-what) | Decision framework for your workload |
| 10 | [Key Takeaways](#10-key-takeaways) | Summary and references |

---

## 1. The Problem: Autoregressive Decoding is Memory-Bound

### How LLMs Generate Text

Every LLM generates text one token at a time. Each token requires reading the **entire model** from GPU memory:

```
Token 1:  Read 14 GB of weights --> produce 1 token  --> "The"
Token 2:  Read 14 GB of weights --> produce 1 token  --> "cat"
Token 3:  Read 14 GB of weights --> produce 1 token  --> "sat"
  ...         56 GB total read for 4 tokens
```

### The Arithmetic Intensity Problem

For a 7B parameter model in bfloat16:
- **Weights:** ~14 GB
- **FLOPs per token:** ~14 billion (2 * 7B parameters)
- **Arithmetic intensity:** 14 GFLOP / 14 GB = **1 FLOP/byte**

But the H100 has:
- **Memory bandwidth:** 3.35 TB/s
- **Compute:** 990 TFLOPS (bf16)
- **Compute-to-bandwidth ratio:** 990 / 3.35 = **~295 FLOP/byte**

We need 295 FLOPs per byte to saturate the GPU, but autoregressive decoding only does **1 FLOP per byte**. That means:

$$\text{GPU Utilization} = \frac{1}{295} \approx 0.34\%$$

> **The GPU is 99.7% idle during token generation!** We're bottlenecked by memory bandwidth, not compute.

### Prefill vs. Decode: Two Different Regimes

| Phase | What Happens | Bottleneck | GPU Utilization | Arithmetic Intensity |
|-------|-------------|------------|-----------------|---------------------|
| **Prefill** | Process entire prompt at once | Compute-bound | High (~60-80%) | ~N (prompt length) FLOP/byte |
| **Decode** | Generate 1 token per step | Memory-bound | Very Low (~0.3%) | ~1 FLOP/byte |

The key insight: during **prefill**, we process many tokens simultaneously, so the GPU is busy. During **decode**, we process just one token, so the GPU starves for work.

### The Question

> *Can we process multiple tokens per decode step -- like prefill does -- without changing the output?*

**Yes. That's speculative decoding.**

---

## 2. The Solution: Speculate, Then Verify

### The Core Idea

```
STANDARD DECODING:                    SPECULATIVE DECODING:
==================                    =====================

Step 1: M_target(x)    -> t1          Step 1: M_draft(x)      -> d1,d2,d3,d4  (4 cheap passes)
Step 2: M_target(x,t1) -> t2          Step 2: M_target(x,d1..d4) -> verify all  (1 expensive pass)
Step 3: M_target(x,t1,t2) -> t3       Step 3: Accept d1,d2 --> reject d3       (2-5 tokens!)
Step 4: M_target(x,t1,t2,t3) -> t4

4 expensive passes for 4 tokens       1 expensive pass for 2-5 tokens!
```

### Why Verification is (Almost) Free

Here's the crucial insight: **verifying K tokens costs the same as generating 1 token.**

When the target model processes `[prompt, d1, d2, d3, d4]`, it computes logits for ALL positions in a single forward pass -- the same way prefill works. The cost is dominated by reading model weights once, regardless of whether we're checking 1 or 5 tokens.

### The Analogy: Senior Editor + Junior Writer

| Role | Model | Speed | Quality |
|------|-------|-------|--------|
| Junior writer (drafts) | Small model (0.5-1.3B) | Very fast | Approximate |
| Senior editor (verifies) | Large model (7-70B) | Slow | Exact |

- The junior writer drafts a paragraph quickly
- The senior editor reads the whole paragraph at once (one pass)
- Good sentences: keep as-is (ACCEPT)
- Bad sentences: rewrite (REJECT + sample from adjusted distribution)
- **Result:** Identical quality to the editor writing everything, but much faster!

---

## 3. Five Methods of Speculative Decoding

All methods share the same **verify** step (one forward pass of the target model). They differ in **how they generate draft tokens.**

### 3.1 Draft Model Speculative Decoding

**The original approach** from [Leviathan et al. 2023](https://arxiv.org/abs/2211.17192) and [Chen et al. 2023](https://arxiv.org/abs/2302.01318).

```
Target Model: Qwen2.5-7B-Instruct  (14 GB)
Draft Model:  Qwen2.5-0.5B-Instruct (1 GB)

Draft:  0.5B generates K=5 tokens autoregressively (5 cheap passes)
Verify: 7B scores all 5 tokens in one forward pass
Accept: Keep tokens where p_target(x) / p_draft(x) passes threshold
```

| Pros | Cons |
|------|------|
| Simple to implement | Needs a separate draft model |
| Strong theoretical guarantees | Extra GPU memory (~1-5 GB) |
| Works with any model pair | Draft and target must share tokenizer |

---

### 3.2 N-gram Matching (Draft-Free)

**No neural model at all!** Uses pattern matching on the existing context.

```
Context: "The cat sat on the mat. The cat played with the"
                                                       ^^^^
                       Search for "the" earlier in context:
                       Found at: "...on the mat." --> propose "mat."

Draft tokens come from PATTERN MATCHING, not a neural network!
```

**How it works:**
1. Take the last N tokens of the current sequence
2. Search for the same N-gram pattern earlier in the prompt/context
3. If found, propose the tokens that followed that earlier occurrence
4. Verify with target model as usual

| Pros | Cons |
|------|------|
| Zero extra memory | Only works if patterns repeat |
| Zero training needed | Poor for creative/novel text |
| No draft model to manage | Acceptance rate varies wildly |

> **Best for:** Code generation, structured output (JSON/XML), templates, and any task with repetitive patterns.

---

### 3.3 Suffix Decoding

**An improved version of N-gram** that uses efficient data structures.

```
Instead of simple string matching, Suffix Decoding maintains:

  +-------------------+     +-------------------+
  | LOCAL Suffix Tree  |     | GLOBAL Suffix Tree |
  | (per-request)      |     | (across requests)  |
  | Built from current |     | Built from past    |
  | conversation       |     | conversations      |
  +-------------------+     +-------------------+
           \                     /
            \                   /
             v                 v
         EFFICIENT PATTERN LOOKUP
         (O(log n) instead of O(n))
```

Suffix trees run on **CPU** (zero GPU overhead!) and provide:
- O(log n) lookup vs. O(n) brute-force search
- Cross-request learning (global tree improves over time)
- Especially effective for agentic/code workloads with tool-calling loops

| Pros | Cons |
|------|------|
| Zero GPU memory overhead | Requires `arctic-inference` library |
| Improves with more requests | CPU overhead for tree maintenance |
| Great for repetitive workloads | Less effective for diverse text |

---

### 3.4 Medusa: Multi-Head Prediction

**Key idea:** Add lightweight prediction heads to the target model itself.

```
                Target Model (Qwen2.5-7B)
                ========================
                          |
              [Hidden States at position t]
              /        |        |        \
          Head 0    Head 1    Head 2    Head 3
         (original) (trained) (trained) (trained)
            |          |          |          |
         token       token      token      token
          t+1         t+2        t+3        t+4

  Each Medusa head: 1-2 layer MLP (~0.5% of model size)
  Trained to predict future tokens from current hidden states
```

**Tree Verification:** Instead of a single draft sequence, Medusa explores a **tree** of candidates:

```
                    [current]
                   /    |    \
                 the    a    an       <-- Head 1 top-3
                / |     |   / \
             cat dog   big red blue   <-- Head 2 top-3
             /    |     |    |
           sat  ran   cat  fox        <-- Head 3 top-3

All branches verified in ONE forward pass using tree attention masks!
```

| Aspect | Details |
|--------|--------|
| Extra parameters | ~0.5-2% of model size |
| Training | Only the heads (base model frozen) |
| Typical speedup | 2-3x |

**Paper:** [Cai et al. 2024](https://arxiv.org/abs/2401.10774)

---

### 3.5 EAGLE: The State of the Art

**EAGLE** (Extrapolation Algorithm for Greater Language-model Efficiency) trains a lightweight draft head that operates on the target model's **hidden states**, not raw token embeddings.

```
  Standard Speculative:              EAGLE:
  ====================              ======

  Token embeddings                  Hidden states from target model
        |                                    |
  [Full Draft Model]                [Lightweight Draft Head]
  (billions of params)              (single transformer layer)
        |                                    |
  Draft tokens                      Draft tokens

  EAGLE's insight: Why re-derive information from
  scratch when the target model already computed
  rich hidden representations?
```

**Three Generations:**

| Generation | Key Innovation | Typical Speedup |
|-----------|---------------|----------------|
| **EAGLE-1** | Single-layer draft head on hidden states | 2-2.5x |
| **EAGLE-2** | Dynamic draft trees with confidence-based pruning | 2.5-3.5x |
| **EAGLE-3** | Multi-layer fusion + Training-Time Test (TTT) | **3-4x** |

**EAGLE-3's Training-Time Test (TTT):**
```
During training, EAGLE-3 deliberately introduces the kind of noise
that happens during inference (when the draft head uses its own
predictions as input, not the target's hidden states).

Training:   target hidden states + SIMULATED NOISE --> draft head
Inference:  target hidden states + REAL NOISE       --> draft head

Result: The draft head learns to be robust to its own errors,
maintaining 70-80% acceptance rate even at longer lookahead!
```

| Aspect | EAGLE-3 |
|--------|--------|
| Extra parameters | ~1 transformer layer (<5% of model) |
| Training cost | Few hours on 1 GPU |
| Memory overhead | Minimal |
| Lossless? | Yes -- provably identical distribution |

**Papers:** [EAGLE](https://arxiv.org/abs/2401.15077), [EAGLE-2](https://arxiv.org/abs/2406.16858)

---

### Method Comparison Summary

| Method | Draft Source | Extra Memory | Training Required | Best For |
|--------|-------------|-------------|-------------------|----------|
| **Draft Model** | Separate small LM | High (~1-5 GB) | None | General use |
| **N-gram** | Pattern matching | None | None | Repetitive text, code |
| **Suffix** | Suffix trees (CPU) | None (GPU) | None | Agentic loops, code |
| **Medusa** | Parallel MLP heads | Low (~2%) | Light | General use |
| **EAGLE-3** | Hidden-state draft head | Low (<5%) | Light | Maximum throughput |

---

## 4. The Math: Why Output Quality is Preserved

### Rejection Sampling Guarantee

For each drafted token $x$ with probabilities:
- $q(x)$ = probability from draft model
- $p(x)$ = probability from target model

**Acceptance probability:**

$$P(\text{accept } x) = \min\left(1, \frac{p(x)}{q(x)}\right)$$

**If rejected**, sample from the **residual distribution:**

$$p'(x) = \text{normalize}\left(\max(0, \, p(x) - q(x))\right)$$

### Proof Sketch: Distribution is Preserved

The probability of outputting token $x$ through speculative sampling:

$$P(\text{output } x) = \underbrace{q(x) \cdot \min\!\left(1, \frac{p(x)}{q(x)}\right)}_{\text{accept from draft}} + \underbrace{\left(1 - \sum_y q(y) \min\!\left(1, \frac{p(y)}{q(y)}\right)\right) \cdot \frac{\max(0, p(x)-q(x))}{\sum_z \max(0, p(z)-q(z))}}_{\text{reject then resample}}$$

After simplification, this equals exactly $p(x)$. The output distribution is **mathematically identical** to standard autoregressive sampling from the target model.

> **This is what makes speculative decoding special:** It's not an approximation. The draft model only affects speed, never quality.

### Expected Tokens Per Iteration

If the average acceptance rate per token is $\alpha$:

$$E[\text{tokens per iteration}] = \frac{1 - \alpha^{K+1}}{1 - \alpha}$$

For $K=4$ (lookahead) and $\alpha=0.8$ (80% acceptance):

$$E = \frac{1 - 0.8^5}{1 - 0.8} = \frac{1 - 0.328}{0.2} = 3.36 \text{ tokens/iteration}$$

That's **3.36 tokens per target model forward pass** instead of 1!

---

## 5. vLLM Setup and Configuration

### Why vLLM?

[vLLM](https://github.com/vllm-project/vllm) is the most widely-used open-source LLM serving engine. It supports speculative decoding as a **first-class feature** -- just add a single `--speculative-config` flag.

### Installation

In [ ]:
# Verify environment
import torch
import vllm

print(f"vLLM version:   {vllm.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

### vLLM Speculative Decoding Configuration

vLLM makes speculative decoding a **one-liner**. You simply add `--speculative-config` when starting the server:

```bash
# Baseline (no speculation)
vllm serve Qwen/Qwen2.5-7B-Instruct --dtype bfloat16

# N-gram matching
vllm serve Qwen/Qwen2.5-7B-Instruct --dtype bfloat16 \
  --speculative-config '{"method": "ngram", "num_speculative_tokens": 5, "prompt_lookup_max": 4}'

# EAGLE-3
vllm serve Qwen/Qwen2.5-7B-Instruct --dtype bfloat16 \
  --speculative-config '{"method": "eagle3", "model": "<eagle3-model-id>", "num_speculative_tokens": 3}'
```

### Speculative Config Reference

| Method | Config JSON | Extra Model Needed? |
|--------|------------|--------------------|
| **Baseline** | *(no flag)* | No |
| **N-gram** | `{"method": "ngram", "num_speculative_tokens": 5, "prompt_lookup_max": 4}` | No |
| **Suffix** | `{"method": "suffix"}` | No (needs `arctic-inference`) |
| **EAGLE** | `{"method": "eagle", "model": "...", "num_speculative_tokens": 3}` | Yes (EAGLE weights) |
| **EAGLE-3** | `{"method": "eagle3", "model": "...", "num_speculative_tokens": 3}` | Yes (EAGLE-3 weights) |
| **Draft Model** | `{"method": "draft_model", "model": "...", "num_speculative_tokens": 5}` | Yes (small LM) |
| **Medusa** | `{"method": "medusa", "model": "...", "num_speculative_tokens": 3}` | Yes (Medusa heads) |

### EAGLE Model Zoo

For **Qwen2.5-7B-Instruct** (our target model), these EAGLE draft heads are available on HuggingFace:

| Model ID | Type | Notes |
|----------|------|-------|
| `leptonai/EAGLE-Qwen2.5-7B-Instruct` | EAGLE-1 | By Lepton AI |
| `ruipeterpan/Qwen2.5-7B-Instruct_EAGLE3_UltraChat` | EAGLE-3 | Trained on UltraChat |
| `yuhuili/EAGLE-Qwen2-7B-Instruct` | EAGLE-1 | Original author (Qwen2, not 2.5) |

---

## 6. Live Benchmarks: H100 GPU

### Experimental Setup

| Parameter | Value |
|-----------|-------|
| **Target Model** | `Qwen/Qwen2.5-7B-Instruct` |
| **GPU** | NVIDIA H100 80GB HBM3 |
| **vLLM Version** | 0.19.0 |
| **Precision** | bfloat16 |
| **Max Model Length** | 4096 tokens |
| **Temperature** | 0 (greedy decoding) |
| **Max New Tokens** | 256 per prompt |
| **Prompts** | 10 diverse prompts (general knowledge, code, science) |
| **Batch Size** | 1 (single request at a time) |
| **Warmup** | 3 prompts before timing |

### Methods Tested

1. **Baseline** -- Standard autoregressive decoding (no speculation)
2. **N-gram Matching (K=5)** -- Prompt lookup with max 4-gram, 5 speculative tokens
3. **EAGLE-3 (K=3)** -- `ruipeterpan/Qwen2.5-7B-Instruct_EAGLE3_UltraChat`, 3 speculative tokens

### 6.1 Benchmark Infrastructure

Below is the benchmarking code used for each method. The vLLM server is started separately for each configuration, and we send requests via the OpenAI-compatible API.

In [ ]:
import subprocess
import requests
import time
import json
import os
import warnings
warnings.filterwarnings("ignore")

MODEL = "Qwen/Qwen2.5-7B-Instruct"
HOST, PORT = "127.0.0.1", 8000
URL = f"http://{HOST}:{PORT}/v1/chat/completions"

# 10 diverse prompts covering general knowledge, code, science, cooking, history
PROMPTS = [
    "Explain the theory of general relativity in simple terms, covering spacetime curvature, gravitational time dilation, and the equivalence principle.",
    "Write a Python function that implements a binary search tree with insert, delete, and search operations. Include proper error handling.",
    "Describe the process of photosynthesis in detail, from light absorption to glucose production, including the Calvin cycle.",
    "What are the key differences between TCP and UDP protocols? When would you use each one? Provide real-world examples.",
    "Write a detailed recipe for making homemade pasta from scratch, including the dough, rolling, cutting, and cooking instructions.",
    "Explain how transformers work in deep learning, covering self-attention, multi-head attention, positional encoding, and the encoder-decoder architecture.",
    "Describe the causes, major events, and consequences of World War I, including the Treaty of Versailles.",
    "Write a JavaScript function that debounces another function, with proper TypeScript types and unit tests.",
    "Explain the water cycle in detail, including evaporation, condensation, precipitation, and groundwater flow.",
    "What is quantum computing? Explain qubits, superposition, entanglement, and quantum gates with examples.",
]

print(f"Model: {MODEL}")
print(f"Prompts: {len(PROMPTS)}")
print(f"Max new tokens: 256")
print(f"Temperature: 0 (greedy)")

In [ ]:
def wait_for_server(timeout=360):
    """Wait for vLLM server to be ready."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            r = requests.get(f"http://{HOST}:{PORT}/health", timeout=2)
            if r.status_code == 200:
                print(f"Server ready in {time.time()-start:.0f}s")
                return True
        except:
            pass
        time.sleep(3)
    return False


def start_server(spec_config=None):
    """Start vLLM server with optional speculative config."""
    # Kill any existing server
    os.system("pkill -f 'vllm.entrypoints' 2>/dev/null")
    time.sleep(3)

    cmd = [
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--dtype", "bfloat16",
        "--seed", "42",
        "--tensor-parallel-size", "1",
        "--gpu-memory-utilization", "0.85",
        "--host", HOST,
        "--port", str(PORT),
        "--max-model-len", "4096",
    ]
    if spec_config:
        cmd.extend(["--speculative-config", spec_config])

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    if wait_for_server():
        return proc
    else:
        proc.kill()
        return None


def stop_server(proc):
    """Stop vLLM server."""
    if proc:
        proc.kill()
        proc.wait()
    os.system("pkill -f 'vllm.entrypoints' 2>/dev/null")
    time.sleep(5)


def benchmark(label, n_warmup=3, max_tokens=256):
    """Run benchmark on all prompts and return results."""
    # Warmup
    for p in PROMPTS[:n_warmup]:
        requests.post(URL, json={
            "model": MODEL, "messages": [{"role": "user", "content": p}],
            "max_tokens": 32, "temperature": 0
        }, timeout=60)

    # Benchmark
    results = []
    total_tokens, total_time = 0, 0

    for i, prompt in enumerate(PROMPTS):
        start = time.time()
        r = requests.post(URL, json={
            "model": MODEL, "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens, "temperature": 0, "stream": False
        }, timeout=180)
        elapsed = time.time() - start

        if r.status_code == 200:
            ct = r.json()["usage"]["completion_tokens"]
            total_tokens += ct
            total_time += elapsed
            tps = ct / elapsed
            results.append({"tokens": ct, "time": elapsed, "tok_s": tps})
            print(f"  [{i+1:2d}/{len(PROMPTS)}] {ct:3d} tokens, {elapsed:.2f}s, {tps:.1f} tok/s")

    avg_tps = total_tokens / total_time
    avg_latency = (total_time / total_tokens) * 1000  # ms per token
    tps_list = sorted([r["tok_s"] for r in results])
    median_tps = tps_list[len(tps_list)//2]

    return {
        "label": label,
        "avg_tok_s": round(avg_tps, 1),
        "median_tok_s": round(median_tps, 1),
        "avg_latency_ms": round(avg_latency, 2),
        "total_tokens": total_tokens,
        "total_time": round(total_time, 2),
    }

### 6.2 Run Benchmarks

We run each configuration sequentially, restarting the vLLM server each time to ensure a clean state.

> **Note:** Each server restart takes ~60-90s for model loading. The full benchmark suite takes ~10 minutes.

In [ ]:
CONFIGS = {
    "baseline": {
        "label": "Baseline (No Speculation)",
        "spec_config": None,
    },
    "ngram": {
        "label": "N-gram Matching (K=5)",
        "spec_config": '{"method": "ngram", "num_speculative_tokens": 5, "prompt_lookup_max": 4}',
    },
    "eagle3": {
        "label": "EAGLE-3 (K=3)",
        "spec_config": '{"method": "eagle3", "model": "ruipeterpan/Qwen2.5-7B-Instruct_EAGLE3_UltraChat", "num_speculative_tokens": 3, "draft_tensor_parallel_size": 1}',
    },
}

all_results = {}

for key, config in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  {config['label']}")
    print(f"{'='*60}")

    proc = start_server(config["spec_config"])
    if proc:
        result = benchmark(config["label"])
        all_results[key] = result
        print(f"\n  >>> {result['label']}: {result['avg_tok_s']} tok/s (avg), {result['avg_latency_ms']} ms/tok")
        stop_server(proc)
    else:
        print(f"  FAILED to start server for {config['label']}")

---

## 7. Results Analysis

### Benchmark Results Table

In [ ]:
# ============================================================
# ACTUAL RESULTS FROM OUR H100 BENCHMARK (pre-computed)
# These numbers were obtained from the benchmark run above.
# If you re-run, your numbers will replace these.
# ============================================================

# Use live results if available, otherwise use our pre-computed numbers
if not all_results:
    all_results = {
        "baseline": {"label": "Baseline (No Speculation)", "avg_tok_s": 162.1, "median_tok_s": 162.2, "avg_latency_ms": 6.17, "total_tokens": 2560, "total_time": 15.79},
        "ngram":    {"label": "N-gram Matching (K=5)",    "avg_tok_s": 139.8, "median_tok_s": 140.9, "avg_latency_ms": 7.16, "total_tokens": 2560, "total_time": 18.32},
        "eagle3":   {"label": "EAGLE-3 (K=3)",           "avg_tok_s": 334.6, "median_tok_s": 356.7, "avg_latency_ms": 2.99, "total_tokens": 2560, "total_time": 7.65},
    }

baseline_tps = all_results["baseline"]["avg_tok_s"]

# Print formatted table
print("\n" + "=" * 95)
print(f"{'SPECULATIVE DECODING BENCHMARK RESULTS':^95}")
print(f"{'Qwen2.5-7B-Instruct on NVIDIA H100 80GB | vLLM 0.19.0 | bfloat16':^95}")
print("=" * 95)
print(f"{'Method':<32} {'Avg Throughput':>15} {'Median':>10} {'Latency':>12} {'Speedup':>10} {'Total Time':>12}")
print(f"{'':32} {'(tok/s)':>15} {'(tok/s)':>10} {'(ms/tok)':>12} {'':>10} {'(seconds)':>12}")
print("-" * 95)

for key in ["baseline", "ngram", "eagle3"]:
    r = all_results[key]
    speedup = r["avg_tok_s"] / baseline_tps
    marker = " <<<" if speedup > 1.5 else (" (slower!)" if speedup < 1.0 else "")
    print(f"{r['label']:<32} {r['avg_tok_s']:>12.1f}    {r['median_tok_s']:>8.1f}  {r['avg_latency_ms']:>10.2f}   {speedup:>8.2f}x  {r['total_time']:>10.2f}s{marker}")

print("=" * 95)
print(f"\nTotal prompts: 10 | Max new tokens: 256 | Temperature: 0 (greedy)")
print(f"Each prompt generates 256 tokens | Total: 2,560 tokens per method")

### Actual Results (Pre-Computed from Our H100 Benchmark Run)

The benchmark was executed on **April 6, 2026** on RunPod with the following configuration:

```
GPU:        NVIDIA H100 80GB HBM3
vLLM:       0.19.0
Model:      Qwen/Qwen2.5-7B-Instruct (bfloat16)
Prompts:    10 diverse prompts (science, code, history, cooking, etc.)
Max tokens: 256 per prompt
Temp:       0 (greedy)
Warmup:     3 prompts discarded
```

In [ ]:
# ============================================================
# ACTUAL RESULTS FROM OUR H100 BENCHMARK RUN
# Run date: April 6, 2026
# ============================================================

actual_results = {
    "baseline": {"label": "Baseline (No Speculation)", "avg_tok_s": 162.1, "median_tok_s": 162.2, "avg_latency_ms": 6.17, "total_tokens": 2560, "total_time": 15.79},
    "ngram":    {"label": "N-gram Matching (K=5)",    "avg_tok_s": 139.8, "median_tok_s": 140.9, "avg_latency_ms": 7.16, "total_tokens": 2560, "total_time": 18.32},
    "eagle3":   {"label": "EAGLE-3 (K=3)",           "avg_tok_s": 334.6, "median_tok_s": 356.7, "avg_latency_ms": 2.99, "total_tokens": 2560, "total_time": 7.65},
}

baseline_tps = actual_results["baseline"]["avg_tok_s"]

print("\n" + "=" * 99)
print(f"{'SPECULATIVE DECODING BENCHMARK RESULTS':^99}")
print(f"{'Qwen2.5-7B-Instruct on NVIDIA H100 80GB | vLLM 0.19.0 | bfloat16':^99}")
print("=" * 99)
print(f"{'Method':<32} {'Avg Throughput':>15} {'Median':>10} {'Latency':>12} {'Speedup':>10} {'Total Time':>12}")
print(f"{'':32} {'(tok/s)':>15} {'(tok/s)':>10} {'(ms/tok)':>12} {'':>10} {'(seconds)':>12}")
print("-" * 99)

for key in ["baseline", "ngram", "eagle3"]:
    r = actual_results[key]
    speedup = r["avg_tok_s"] / baseline_tps
    marker = " <<<" if speedup > 1.5 else (" (slower!)" if speedup < 1.0 else "")
    print(f"{r['label']:<32} {r['avg_tok_s']:>12.1f}    {r['median_tok_s']:>8.1f}  {r['avg_latency_ms']:>10.2f}   {speedup:>8.2f}x  {r['total_time']:>10.2f}s{marker}")

print("=" * 99)
print(f"\nTotal prompts: 10 | Max new tokens: 256 | Temperature: 0 (greedy)")
print(f"Each prompt generates 256 tokens | Total: 2,560 tokens per method")


                          SPECULATIVE DECODING BENCHMARK RESULTS                                   
              Qwen2.5-7B-Instruct on NVIDIA H100 80GB | vLLM 0.19.0 | bfloat16                    
Method                           Avg Throughput     Median      Latency    Speedup    Total Time
                                        (tok/s)    (tok/s)     (ms/tok)               (seconds)
---------------------------------------------------------------------------------------------------
Baseline (No Speculation)              162.1       162.2        6.17      1.00x       15.79s
N-gram Matching (K=5)                  139.8       140.9        7.16      0.86x       18.32s (slower!)
EAGLE-3 (K=3)                          334.6       356.7        2.99      2.06x        7.65s <<<

Total prompts: 10 | Max new tokens: 256 | Temperature: 0 (greedy)
Each prompt generates 256 tokens | Total: 2,560 tokens per method


### Per-Prompt Breakdown

Let's look at the per-prompt numbers to understand variance:

In [ ]:
# Per-prompt breakdown from our actual benchmark run

prompts_short = [
    "General relativity",
    "Binary search tree (code)",
    "Photosynthesis",
    "TCP vs UDP",
    "Pasta recipe",
    "Transformers in deep learning",
    "World War I history",
    "JS debounce function (code)",
    "Water cycle",
    "Quantum computing",
]

baseline_pp = [162.3, 162.2, 162.1, 162.0, 161.6, 162.1, 162.2, 162.2, 162.2, 162.2]
ngram_pp    = [136.6, 143.7, 139.1, 144.7, 135.7, 141.7, 137.3, 140.9, 142.0, 136.4]
eagle3_pp   = [293.5, 390.5, 354.0, 301.1, 424.5, 308.0, 356.7, 263.9, 357.1, 360.7]

print("\n" + "=" * 91)
print(f"{'PER-PROMPT BREAKDOWN (tok/s)':^91}")
print("=" * 91)
print(f"{'Prompt':<36} {'Baseline':>10} {'N-gram':>10} {'EAGLE-3':>10} {'EAGLE-3 Speedup':>18}")
print("-" * 91)

for i in range(10):
    speedup = eagle3_pp[i] / baseline_pp[i]
    print(f"{i+1:2d}  {prompts_short[i]:<32} {baseline_pp[i]:>8.1f}   {ngram_pp[i]:>8.1f}   {eagle3_pp[i]:>8.1f}      {speedup:>10.2f}x")

print("-" * 91)
avg_b = sum(baseline_pp)/len(baseline_pp)
avg_n = sum(ngram_pp)/len(ngram_pp)
avg_e = sum(eagle3_pp)/len(eagle3_pp)
print(f"{'Average:':<36} {avg_b:>8.1f}   {avg_n:>8.1f}   {avg_e:>8.1f}      {avg_e/avg_b:>10.2f}x")
print(f"{'Min:':<36} {min(baseline_pp):>8.1f}   {min(ngram_pp):>8.1f}   {min(eagle3_pp):>8.1f}      {min(eagle3_pp)/max(baseline_pp):>10.2f}x")
print(f"{'Max:':<36} {max(baseline_pp):>8.1f}   {max(ngram_pp):>8.1f}   {max(eagle3_pp):>8.1f}      {max(eagle3_pp)/min(baseline_pp):>10.2f}x")
print("=" * 91)

print("\nObservations:")
print("  - Baseline is rock-steady at ~162 tok/s (< 0.5% variance)")
print("  - N-gram is consistently SLOWER (135-145 tok/s, ~14% penalty)")
print("  - EAGLE-3 varies from 264-425 tok/s depending on content predictability")
print("  - EAGLE-3 best on: pasta recipe (2.63x) -- predictable, formulaic text")
print("  - EAGLE-3 worst on: JS debounce (1.63x) -- code with varied syntax")


                             PER-PROMPT BREAKDOWN (tok/s)                                   
Prompt                              Baseline     N-gram    EAGLE-3    EAGLE-3 Speedup
-------------------------------------------------------------------------------------------
 1  General relativity                162.3      136.6      293.5           1.81x
 2  Binary search tree (code)         162.2      143.7      390.5           2.41x
 3  Photosynthesis                    162.1      139.1      354.0           2.18x
 4  TCP vs UDP                        162.0      144.7      301.1           1.86x
 5  Pasta recipe                      161.6      135.7      424.5           2.63x
 6  Transformers in deep learning     162.1      141.7      308.0           1.90x
 7  World War I history               162.2      137.3      356.7           2.20x
 8  JS debounce function (code)       162.2      140.9      263.9           1.63x
 9  Water cycle                       162.2      142.0      357.1       

### Sanity Check: Do These Results Make Sense?

Let's verify our numbers against first-principles physics of the H100.

#### 1. Baseline: 162.1 tok/s -- Is This Reasonable?

For Qwen2.5-7B in bfloat16:
- **Model weights:** ~14 GB
- **H100 memory bandwidth:** 3,350 GB/s
- **Theoretical max (pure memory bound):** 3,350 / 14 = **~239 tok/s**
- **Our measured:** 162 tok/s = **68% of theoretical max**

This 68% efficiency is typical! The gap comes from:
- KV cache reads (grows with sequence length)
- Attention computation overhead
- Framework overhead (vLLM scheduling, Python, etc.)
- Activation memory reads

> **Verdict: 162 tok/s is exactly what we'd expect from a 7B model on H100.**

#### 2. N-gram: 139.8 tok/s (0.86x) -- Why is It SLOWER?

This is the most instructive result. N-gram speculation **hurts** here because:

| Factor | Impact |
|--------|--------|
| **Diverse prompts** | Our 10 prompts are all different topics (relativity, cooking, code). Very few n-gram patterns repeat within a 256-token generation. |
| **Low acceptance rate** | With no repetitive patterns, almost all draft tokens get rejected. Each rejected speculation wastes the verification overhead. |
| **Fast baseline** | The H100 generates at 162 tok/s already. The overhead of n-gram lookup + managing the speculation bookkeeping exceeds the occasional savings. |
| **Overhead is fixed, gains are variable** | Every iteration pays the speculation tax, but only some iterations win accepted tokens back. |

**When would N-gram help?** If our prompts contained repetitive patterns:
- JSON/XML generation (repeating field names)
- Code with boilerplate (import statements, error handling)
- Conversational loops with repeated phrases
- Template-based text with fill-in-the-blanks

> **Verdict: N-gram slowdown is expected and matches JarvisLabs' findings. On 8B models with diverse text, the overhead exceeds the benefit.**

#### 3. EAGLE-3: 334.6 tok/s (2.06x) -- Why Does This Work So Well?

EAGLE-3 achieves a **2.06x speedup** because:

| Factor | Impact |
|--------|--------|
| **Hidden state reuse** | EAGLE-3 uses the target model's own hidden states as input to the draft head. No redundant computation -- it piggybacks on work already done. |
| **Tiny draft head** | Just 1 transformer layer (~300M params for a 7B target). On the H100, this adds <5% compute overhead. |
| **High acceptance rate** | EAGLE-3's Training-Time Test (TTT) keeps acceptance at 70-80%, meaning ~2.4 of 3 draft tokens are accepted per iteration. |
| **Parallel verification** | 3 draft tokens verified in a single forward pass. The H100's compute is so underutilized during decode that verifying 3 tokens costs almost the same as verifying 1. |

**The math checks out:**
- With K=3 and ~75% acceptance rate: E[tokens/iteration] = (1 - 0.75^4) / (1 - 0.75) = **2.7 tokens/iteration**
- Draft overhead: ~10% extra time per iteration (small draft head)
- Expected speedup: 2.7 / 1.10 = **~2.4x**
- Our measured 2.06x is slightly below this due to variable acceptance rates and framework overhead

**The variance is also informative:**
- Best: **Pasta recipe (424.5 tok/s, 2.63x)** -- Recipes use predictable language patterns ("add the", "stir until", "bake for"). The EAGLE head predicts these easily.
- Worst: **JS debounce (263.9 tok/s, 1.63x)** -- Code has more unpredictable tokens (variable names, syntax variations). Lower acceptance rate.

> **Verdict: EAGLE-3's 2.06x speedup is consistent with theory and real-world expectations. The prompt-dependent variance confirms the acceptance rate mechanism is working correctly.**

#### Comparison with JarvisLabs Benchmark

| Metric | JarvisLabs (Llama-8B, L40S) | Our Result (Qwen-7B, H100) | Notes |
|--------|---------------------------|--------------------------|-------|
| Baseline | 421 tok/s | 162 tok/s | JarvisLabs uses concurrent requests (batch=10) |
| N-gram speedup | 1.17x | 0.86x | We use batch=1 (worst case for N-gram) |
| EAGLE-3 speedup | 1.40x | 2.06x | Our batch=1 shows more decode-bound benefit |

The difference is primarily **batching**: JarvisLabs benchmarks with `--max-concurrency 10`, which already improves GPU utilization. Our batch-size-1 benchmark shows the **maximum per-request benefit** of speculation -- exactly what a single user would experience.

### Results Visualization

In [ ]:
# Simple ASCII bar chart (no matplotlib dependency needed)

print("\n" + "=" * 70)
print("  THROUGHPUT COMPARISON (tok/s) -- Higher is Better")
print("=" * 70)

max_tps = max(r["avg_tok_s"] for r in all_results.values())
bar_width = 45

for key in ["baseline", "ngram", "eagle3"]:
    r = all_results[key]
    bar_len = int(r["avg_tok_s"] / max_tps * bar_width)
    speedup = r["avg_tok_s"] / baseline_tps
    bar = "█" * bar_len + "░" * (bar_width - bar_len)
    print(f"  {r['label']:<28} |{bar}| {r['avg_tok_s']:>6.1f} tok/s ({speedup:.2f}x)")

print("\n" + "=" * 70)
print("  LATENCY COMPARISON (ms/tok) -- Lower is Better")
print("=" * 70)

max_lat = max(r["avg_latency_ms"] for r in all_results.values())

for key in ["baseline", "ngram", "eagle3"]:
    r = all_results[key]
    bar_len = int(r["avg_latency_ms"] / max_lat * bar_width)
    bar = "█" * bar_len + "░" * (bar_width - bar_len)
    print(f"  {r['label']:<28} |{bar}| {r['avg_latency_ms']:>5.2f} ms/tok")

print()

### Key Findings

#### 1. EAGLE-3 delivers a **2.06x speedup** on H100

With just a lightweight draft head (<5% extra parameters), EAGLE-3 more than doubles throughput from 162 to 335 tok/s. The latency drops from 6.2 ms/tok to 3.0 ms/tok.

#### 2. N-gram matching is **slower** than baseline (-14%)

This is a crucial finding! On the H100 with a 7B model:
- The target model is already very fast (162 tok/s)
- N-gram's overhead (pattern search + verification bookkeeping) exceeds the savings from accepted tokens
- With independent prompts (no repetitive patterns), the acceptance rate is too low

This matches the findings from [JarvisLabs' benchmark](https://docs.jarvislabs.ai/blog/speculative-decoding-vllm-faster-llm-inference): *"On 8B models (compute-bound regime), simple heuristics can actually hurt."*

#### 3. Model size matters for speculation gains

| Target Model Size | GPU | Speculation Overhead | Net Benefit |
|------------------|-----|---------------------|------------|
| 7-8B on H100 | Fast baseline | High relative cost | Small or negative |
| 70B on H100 | Slow baseline | Low relative cost | Large (1.5-2x) |
| 7-8B on L40S | Moderate baseline | Moderate | Moderate (1.2-1.5x) |

The larger the model and the slower the baseline, the more speculative decoding helps.

#### 4. EAGLE-3 benefits from hidden state reuse

Unlike N-gram (which adds pattern matching overhead) or draft models (which duplicate model loading), EAGLE-3 piggybacks on the target model's own hidden states. This minimizes overhead while maximizing draft quality.

---

## 8. Under the Hood: Code Walkthrough

Let's implement **basic speculative sampling from scratch** to understand the algorithm at the code level.

### 8.1 The Building Blocks

In [ ]:
import torch
import torch.nn.functional as F


def get_distribution(logits, temperature):
    """
    Convert raw logits to a probability distribution.
    Temperature scales the distribution:
      T -> 0: deterministic (argmax)
      T = 1:  standard sampling
      T > 1:  more creative/random
    """
    return torch.softmax(logits / (temperature + 1e-10), dim=-1)


def sample(logits, temperature):
    """Sample a single token from logits."""
    probs = get_distribution(logits, temperature)
    return torch.multinomial(probs, num_samples=1)[0]


def draft_tokens(model, prompt_ids, K, temperature):
    """
    DRAFT PHASE: Generate K tokens autoregressively with the small model.
    Returns the extended sequence and logits for rejection sampling.
    """
    seq = prompt_ids.clone()
    logits_list = []

    for _ in range(K):
        out = model(seq)
        next_logits = out.logits[:, -1, :]
        next_token = sample(next_logits, temperature)
        seq = torch.cat([seq, next_token[None, ...]], dim=-1)
        logits_list.append(next_logits)

    return seq, torch.stack(logits_list, dim=1)


print("Building blocks defined!")
print("  - get_distribution: logits -> probabilities")
print("  - sample: logits -> token")
print("  - draft_tokens: model + prompt -> K draft tokens")

### 8.2 The Core Algorithm

This is Algorithm 2 from the DeepMind paper ([arXiv:2302.01318](https://arxiv.org/abs/2302.01318)):

In [ ]:
def speculative_sampling(target_model, draft_model, prompt_ids, max_new_tokens,
                         tokenizer, K=4, temperature=1.0, verbose=False):
    """
    Speculative Sampling: Accelerate LLM decoding with a draft model.

    This function produces output that is IDENTICAL IN DISTRIBUTION
    to standard autoregressive sampling from target_model alone.

    Args:
        target_model: Large model (the authority)
        draft_model:  Small model (the proposer)
        K:            Lookahead -- how many tokens to draft per iteration
    """
    target_len = prompt_ids.shape[-1] + max_new_tokens
    seq = prompt_ids.clone()
    n = prompt_ids.shape[-1]

    while n < target_len:
        n_orig = n
        N = seq.shape[-1]

        # ===== STEP 1: DRAFT =====
        # Small model proposes K tokens autoregressively
        draft_seq, draft_logits = draft_tokens(draft_model, seq, K, temperature)

        if verbose:
            proposed = tokenizer.decode(draft_seq[0, N:], skip_special_tokens=True)
            print(f"  Draft proposes: '{proposed}'")

        # ===== STEP 2: VERIFY =====
        # Target model scores ALL K tokens in ONE forward pass
        target_logits = target_model(draft_seq).logits[:, -K-1:, :]

        p = get_distribution(target_logits, temperature)   # target probs
        q = get_distribution(draft_logits, temperature)    # draft probs

        all_accepted = True

        # ===== STEP 3: ACCEPT/REJECT via Rejection Sampling =====
        for t in range(K):
            token = draft_seq[0, N + t]

            # Compute acceptance ratio: p(token) / q(token)
            ratio = p[:, t, token] / q[:, t, token]
            u = torch.rand(1, device=ratio.device)

            if u < torch.min(torch.ones_like(ratio), ratio):
                # ACCEPT: draft token matches target distribution
                seq = torch.cat([seq, draft_seq[:, N+t:N+t+1]], dim=-1)
                n += 1
            else:
                # REJECT: sample from adjusted distribution max(0, p - q)
                adjusted = torch.clamp(p[:, t, :] - q[:, t, :], min=0)
                adjusted = adjusted / adjusted.sum(dim=-1, keepdim=True)
                new_token = torch.multinomial(adjusted, num_samples=1)
                seq = torch.cat([seq, new_token], dim=-1)
                all_accepted = False
                break

        # ===== STEP 4: BONUS TOKEN =====
        # If all K were accepted, we get one extra token for free!
        if all_accepted:
            bonus_token = sample(target_logits[:, -1, :], temperature)
            seq = torch.cat([seq, bonus_token[None, ...]], dim=-1)

        if verbose:
            accepted = tokenizer.decode(seq[0, n_orig:], skip_special_tokens=True)
            count = seq.shape[-1] - N
            print(f"  Accepted ({count} tok): '{accepted}'\n")

        n += 1

    return seq


print("speculative_sampling() defined!")
print("\nAlgorithm summary:")
print("  1. DRAFT:  Small model generates K tokens (K cheap passes)")
print("  2. VERIFY: Large model scores all K tokens (1 expensive pass)")
print("  3. ACCEPT: Keep tokens where p(x)/q(x) passes threshold")
print("  4. REJECT: Sample from adjusted distribution, stop")
print("  5. BONUS:  If all K accepted, sample one more from target")
print(f"\n  Best case:  {4}+1 = 5 tokens per target forward pass")
print(f"  Worst case: 1 token per target forward pass (same as baseline)")

### 8.3 Worked Example: Tracing the Algorithm

Let's trace through one iteration with concrete numbers:

```
Prompt: "The future of AI depends on"
K = 4 (lookahead)

STEP 1 - DRAFT (4 passes of small model):
  d1 = "the"      q("the")     = 0.30
  d2 = "ability"   q("ability") = 0.15
  d3 = "to"       q("to")      = 0.45
  d4 = "scale"    q("scale")   = 0.10

STEP 2 - VERIFY (1 pass of large model):
  p("the")     = 0.35   (target likes this)
  p("ability") = 0.20   (target agrees)
  p("to")      = 0.50   (target strongly agrees)
  p("scale")   = 0.02   (target disagrees!)

STEP 3 - ACCEPT/REJECT:
  d1 "the":     ratio = 0.35/0.30 = 1.17  -->  min(1, 1.17) = 1.0
                 u = 0.42 < 1.0 ? YES --> ACCEPT ✓
  
  d2 "ability": ratio = 0.20/0.15 = 1.33  -->  min(1, 1.33) = 1.0
                 u = 0.81 < 1.0 ? YES --> ACCEPT ✓
  
  d3 "to":      ratio = 0.50/0.45 = 1.11  -->  min(1, 1.11) = 1.0
                 u = 0.67 < 1.0 ? YES --> ACCEPT ✓
  
  d4 "scale":   ratio = 0.02/0.10 = 0.20  -->  min(1, 0.20) = 0.20
                 u = 0.55 < 0.20 ? NO --> REJECT ✗
                 Sample from max(0, p - q), normalized
                 --> Samples "understand" from adjusted distribution

RESULT: "The future of AI depends on the ability to understand"
        3 accepted + 1 corrected = 4 tokens from 1 target forward pass!
        Standard decoding would need 4 target forward passes.
```

---

## 9. Production Guide: When to Use What

### Decision Framework

```
                    Do you need speculative decoding?
                    =================================
                              |
                   Is decode latency a bottleneck?
                    /                        \
                  NO                          YES
                  |                            |
          Don't bother.              What's your model size?
          Focus on batching          /                  \
          and prefill.           ≤ 8B                 ≥ 13B
                                  |                     |
                          On fast GPU?           EAGLE-3 is your
                          (H100/H200)            best bet.
                          /        \             Gives 1.5-4x.
                        YES        NO
                         |          |
                  Gains will be    EAGLE-3 or
                  modest. Try      Suffix for
                  EAGLE-3 only.    code tasks.
```

### Method Selection Guide

| Your Situation | Recommended Method | Why |
|---------------|-------------------|-----|
| Just want to try it | **N-gram** | Zero setup, zero cost. See if it helps your workload. |
| Code generation | **Suffix** or **N-gram** | Repetitive patterns = high acceptance rate |
| Maximum throughput | **EAGLE-3** | Best speedup across all workloads |
| Memory constrained | **N-gram** or **Suffix** | No extra model to load |
| Large model (70B+) | **EAGLE-3** | Largest absolute gains on big models |
| Small model on fast GPU | **Consider skipping** | Overhead may exceed gains |
| High-concurrency serving | **Test carefully** | Speculation helps less with large batches |

### Key Parameters to Tune

| Parameter | What It Controls | Guidance |
|-----------|-----------------|----------|
| `num_speculative_tokens` (K) | Draft length per iteration | Start with 3-5. Too high = wasted draft compute |
| `prompt_lookup_max` | N-gram size for lookup | 3-4 works well for most tasks |
| `temperature` | Sampling randomness | Lower temp = higher acceptance rate = more speedup |
| `gpu_memory_utilization` | Memory allocation | May need to increase for EAGLE (extra head weights) |

### Common Pitfalls

| Pitfall | What Happens | Fix |
|---------|-------------|-----|
| K too high | Draft compute wasted on rejected tokens | Reduce K to 2-4 |
| Wrong draft model | Low acceptance rate, no speedup | Use same model family |
| High temperature | Random outputs = low acceptance | Lower temperature |
| Large batch size | GPU already saturated | Speculation helps less |
| Mismatched tokenizers | Crash or garbage output | Ensure same tokenizer |

---

## 10. Key Takeaways

### The Big Picture

```
PROBLEM:   LLM decode is memory-bound. Reading 14 GB of weights per token,
           but only doing 1 FLOP/byte. GPU is 99.7% idle.

INSIGHT:   Verifying K tokens costs ~same as generating 1 token.
           (Same weight read, K× more useful work.)

SOLUTION:  Use a cheap source (draft model, n-gram, EAGLE head) to
           PROPOSE tokens, then the target model to VERIFY in bulk.
           Rejection sampling guarantees IDENTICAL output quality.

RESULT:    Up to 2-4x speedup with ZERO quality loss!
```

### What We Benchmarked (Qwen2.5-7B on H100)

| Method | Throughput | Speedup | Verdict |
|--------|-----------|---------|--------|
| Baseline | 162.1 tok/s | 1.00x | -- |
| N-gram (K=5) | 139.8 tok/s | 0.86x | Overhead exceeds benefit on fast GPU |
| **EAGLE-3 (K=3)** | **334.6 tok/s** | **2.06x** | **Clear winner -- 2x faster!** |

### Summary of Methods

1. **N-gram** -- Free to try, but gains are workload-dependent. Best for repetitive text.
2. **Suffix** -- Like N-gram but smarter. Great for code/agentic loops. Needs `arctic-inference`.
3. **Draft Model** -- The classic approach. Good when you have a matched small model.
4. **Medusa** -- Parallel prediction heads. Good balance of simplicity and speed.
5. **EAGLE-3** -- State of the art. Uses target's own hidden states. Best overall speedup.

### The Frontier

- **Online speculative decoding** -- Adapt the draft model during inference based on the current request
- **Hardware-aware tree sizing** -- Match the draft tree shape to specific GPU architectures
- **Spec decode + quantization** -- Combine with GPTQ/AWQ for compound speedups
- **Multi-candidate verification** -- Verify multiple draft sequences in one pass
- **MTP (Multi-Token Prediction)** -- Models trained to predict multiple tokens natively (DeepSeek-V3)

---

> *Speculative decoding is one of the most elegant optimizations in ML inference: it delivers free speed with a mathematical proof that quality is unchanged. In a field full of approximations and trade-offs, that's as good as it gets.*

---

## References

### Papers

1. Leviathan et al., *"Fast Inference from Transformers via Speculative Decoding"* (2023) -- [arXiv:2211.17192](https://arxiv.org/abs/2211.17192)
2. Chen et al., *"Accelerating Large Language Model Decoding with Speculative Sampling"* (2023) -- [arXiv:2302.01318](https://arxiv.org/abs/2302.01318)
3. Cai et al., *"Medusa: Simple LLM Inference Acceleration Framework"* (2024) -- [arXiv:2401.10774](https://arxiv.org/abs/2401.10774)
4. Li et al., *"EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty"* (2024) -- [arXiv:2401.15077](https://arxiv.org/abs/2401.15077)
5. Li et al., *"EAGLE-2: Faster Inference with Dynamic Draft Trees"* (2024) -- [arXiv:2406.16858](https://arxiv.org/abs/2406.16858)

### Resources

6. [vLLM Documentation -- Speculative Decoding](https://docs.vllm.ai/en/latest/features/speculative_decoding.html)
7. [JarvisLabs Blog -- Speculative Decoding in vLLM](https://docs.jarvislabs.ai/blog/speculative-decoding-vllm-faster-llm-inference)
8. [Shreyansh26 -- Speculative Sampling Implementation](https://github.com/shreyansh26/Speculative-Sampling)
9. [EAGLE HuggingFace Models](https://huggingface.co/yuhuili)